# Lone-Wolf-Howler — guarded retraining

This notebook checks the source and output-suppression tests, stages only the approved JSONL files, and runs the recovery-aware training pipeline. GPU and Internet must be enabled. Add a Kaggle Secret named `HF_TOKEN` with a Hugging Face token that can publish to your configured model repository. Do not paste tokens into notebook cells.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

WORK = Path('/kaggle/working')
REPO = WORK / 'Shadow-ronin-master'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/yalangisexter-ux/Shadow-ronin-master.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

# Require the publishing secret before spending GPU time.
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add the HF_TOKEN secret in Kaggle Notebook > Add-ons > Secrets, then rerun.')
os.environ['HF_TOKEN'] = hf_token
del hf_token

# Only approved source datasets are staged; avoid recursively mixing unrelated Kaggle inputs.
DATA = WORK / 'lone_wolf_training_data'
DATA.mkdir(parents=True, exist_ok=True)
for name in ('combined_clean_dataset_augmented.jsonl', 'cyber.jsonl'):
    src = REPO / name
    if not src.is_file():
        raise FileNotFoundError(f'Required training dataset missing: {src}')
    shutil.copy2(src, DATA / name)

os.environ['TRAINING_DATA_DIR'] = str(DATA)
os.environ['FORCE_SCRATCH_TRAINING'] = '1'
os.environ['ENABLE_GRPO'] = '0'
os.environ['DISABLE_RAG'] = '1'  # RAG remains disabled for this run.
os.environ.setdefault('MODEL_ID', 'Qwen/Qwen3-1.7B')

# Syntax and output-regression checks before training.
for name in ('Lone-Wolf-Howler-End-to-End-Recovery-First-Fixed.py',
             'Lone-Wolf-Howler-End-to-End-Recovery-First-Test.py',
             'Shadow-Ronin-4B-End-to-End-Recovery-First-Master.py',
             'lone_wolf_inference.py'):
    subprocess.run([sys.executable, '-m', 'py_compile', str(REPO / name)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(REPO / 'tests'), '-v'], check=True)

# Run the fixed pipeline. Failure is surfaced rather than reported as a successful deploy.
subprocess.run([sys.executable, str(REPO / 'Lone-Wolf-Howler-End-to-End-Recovery-First-Fixed.py')],
               cwd=str(REPO), env=os.environ.copy(), check=True)
